In [ ]:
import re
from pathlib import Path
from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"
drive.mount("/content/drive")

if not COMMIT_FILE.is_file() or not DIRTY_FILE.is_file():
    raise RuntimeError("Missing project state markers")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if DIRTY_FILE.read_text(encoding="utf-8").strip() not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file() or len(wheels) != 1:
    raise RuntimeError("Expected locked requirements and one project wheel")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheels[0]}
print("Colab project installation complete")

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab
context = initialize_colab(enable_wandb=False, require_cuda=True)
context

# FLenQA failure concept intervention

For short-correct / long-wrong pairs, choose J-Lens concepts separately for each pair. We restore concepts lost in the long prompt, and inject concepts gained in the failed long prompt.

In [ ]:
import string
import jlens
import pandas as pd
import torch
import transformers
from datasets import load_from_disk
from jlens.hooks import ActivationRecorder

from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.benchmarks.flenqa.lens import ApplyLensRunner, LensRunners, run_prompt
from jlens_reasoning.benchmarks.flenqa.positions import prepare_prompt
from jlens_reasoning.evaluation import evaluate_next_token
from jlens_reasoning.experiments_utils.interventions import LensCoordinatePatcher, jlens_vector, lens_coordinates
from jlens_reasoning.experiments_utils.validation import validate_model_lens

LAYER = 2
TOP_K = 250
SHORT_CTX_SIZE, LONG_CTX_SIZE = 250, 1000
N_PROBLEMS = None
K_VALUES = (1, 3, 5, 10)
ALPHAS = (0.0, 0.5, 1.0)
POSITION_LABEL = "final_prompt"
MAX_SEQ_LEN = 4096

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows)

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, local_files_only=True).to(context.device)
causal_lm.eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
validate_model_lens(model, lens)
assert LAYER in lens.source_layers
runners = LensRunners(ApplyLensRunner(lens, model, True, layers=(LAYER,)), ApplyLensRunner(lens, model, False, layers=(LAYER,)))
unembedding_weight = causal_lm.get_output_embeddings().weight

## Matched prompts and baseline failures

In [ ]:
by_problem = {}
for row in rows:
    by_problem.setdefault(row.problem_id, []).append(row)
problem_ids = [p for p, rs in by_problem.items() if {r.ctx_size_declared for r in rs} >= {SHORT_CTX_SIZE, LONG_CTX_SIZE}][:N_PROBLEMS]
prepared_prompts = prepare_prompts([r for r in rows if r.problem_id in problem_ids])

def nominal_context_size(prompt):
    sizes = {item.ctx_size for item in prompt.provenance}
    if len(sizes) != 1:
        raise ValueError(f"Prompt {prompt.prompt_id} spans context sizes {sizes}")
    return sizes.pop()

pairs = []
for problem_id in problem_ids:
    candidates = [p for p in prepared_prompts if p.problem_id == problem_id]
    short = sorted((p for p in candidates if nominal_context_size(p) == SHORT_CTX_SIZE), key=lambda p: p.canonical_index)[0]
    long = sorted((p for p in candidates if nominal_context_size(p) == LONG_CTX_SIZE), key=lambda p: p.canonical_index)[0]
    if (short.task, short.label) != (long.task, long.label):
        raise ValueError(f"Matched prompts disagree for {problem_id}")
    pairs.append({"problem_id": problem_id, "short": short, "long": long})

prepared_by_id, topk_tables = {}, []
for pair in pairs:
    for prompt in (pair["short"], pair["long"]):
        prepared = prepare_prompt(prompt, tokenizer, max_seq_len=MAX_SEQ_LEN)
        prepared_by_id[prompt.prompt_id] = (prepared, prepared.positions[POSITION_LABEL][0])
        result = run_prompt(prepared, runners=runners, top_k=TOP_K, max_seq_len=MAX_SEQ_LEN, logits_rtol=1e-5, logits_atol=1e-6)
        topk_tables.append(result.batches["topk"].to_pandas())

topk = pd.concat(topk_tables, ignore_index=True).query("lens_kind == 'jacobian' and layer == @LAYER").copy()
topk["rank_score"] = 1 / topk["rank"]
token_text = {int(i): tokenizer.decode([int(i)], clean_up_tokenization_spaces=False) for i in topk.token_id.unique()}
token_raw = {int(i): tokenizer.convert_ids_to_tokens(int(i)) for i in topk.token_id.unique()}

CONTROL_SURFACES = {"yes", "no", "true", "false", "answer", "response", "think", "/think"}
SPECIAL_SURFACES = {s.casefold() for s in tokenizer.all_special_tokens}
def exclusion_reason(token_id):
    text = token_text[token_id]
    raw = str(token_raw[token_id])
    normalized = text.strip().casefold().lstrip("▁Ġ")
    if token_id in set(tokenizer.all_special_ids) or normalized in SPECIAL_SURFACES:
        return "special token"
    if not normalized or all(c in string.punctuation for c in normalized):
        return "formatting/empty"
    if normalized in CONTROL_SURFACES or normalized.strip(string.punctuation) in CONTROL_SURFACES:
        return "answer/control surface"
    if "�" in text or "<0x" in raw or any(ord(c) < 32 for c in text if c not in "\n\t"):
        return "encoding artifact"
    if not re.fullmatch(r"[A-Za-z]+(?:['’-][A-Za-z]+)*", normalized):
        return "not word-like"
    if raw.startswith("##"):
        return "tokenizer continuation"
    return None

topk["excluded_reason"] = [exclusion_reason(int(i)) for i in topk.token_id]
topk["token"] = topk.token_id.map(token_text)
filtered_out = topk[topk.excluded_reason.notna()]
usable = topk[topk.excluded_reason.isna()].copy()
print("Short/long matched pairs:", len(pairs))

def next_token_score(logits, prompt):
    expected = "True" if prompt.label else "False"
    evaluation = evaluate_next_token(logits, expected, tokenizer, top_k=10)
    return {"prediction": evaluation.top1_token, "score": int(evaluation.target_rank == 1), "target_rank": evaluation.target_rank}

def baseline_for(prompt):
    input_ids = torch.tensor([prepared_by_id[prompt.prompt_id][0].input_ids], device=context.device)
    with torch.inference_mode():
        logits = causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]
    return next_token_score(logits, prompt)

failures = []
for pair in pairs:
    short_result, long_result = baseline_for(pair["short"]), baseline_for(pair["long"])
    if short_result["score"] == 1 and long_result["score"] == 0:
        pair = {**pair, "short_baseline": short_result, "long_baseline": long_result}
        failures.append(pair)
print("Short-correct / long-wrong pairs:", len(failures))

## Per-pair concept selection

Reciprocal rank is zero outside the top-250 readout. Positive score differences select concepts active in only one member of the pair.

In [ ]:
ranked_by_prompt = usable.set_index(["prompt_id", "token_id"])["rank_score"]
selection_rows, selected_ids = [], set()
for pair in failures:
    short_id, long_id = pair["short"].prompt_id, pair["long"].prompt_id
    ids = set(usable[usable.prompt_id.isin([short_id, long_id])].token_id.astype(int))
    candidates = pd.DataFrame([{"token_id": i, "short_score": ranked_by_prompt.get((short_id, i), 0), "long_score": ranked_by_prompt.get((long_id, i), 0)} for i in ids])
    candidates["lost_score"] = candidates.short_score - candidates.long_score
    candidates["gained_score"] = candidates.long_score - candidates.short_score
    lost = candidates.query("lost_score > 0").sort_values(["lost_score", "short_score"], ascending=False).head(max(K_VALUES)).copy()
    gained = candidates.query("gained_score > 0").sort_values(["gained_score", "long_score"], ascending=False).head(max(K_VALUES)).copy()
    for kind, selected in (("lost", lost), ("gained", gained)):
        selected_ids.update(selected.token_id.astype(int))
        for rank, row in enumerate(selected.itertuples(), 1):
            selection_rows.append({"problem_id": pair["problem_id"], "kind": kind, "selection_rank": rank, "token_id": int(row.token_id), "token": token_text[int(row.token_id)], "short_score": row.short_score, "long_score": row.long_score, "short_rank": round(1 / row.short_score) if row.short_score else None, "long_rank": round(1 / row.long_score) if row.long_score else None})
selections = pd.DataFrame(selection_rows)
display(selections.groupby("kind").head(10))

def activation_for_prompt(prompt_id):
    prepared, position = prepared_by_id[prompt_id]
    input_ids = torch.tensor([prepared.input_ids], device=context.device)
    with torch.inference_mode(), ActivationRecorder(model.layers, at=(LAYER,)) as recorder:
        causal_lm(input_ids=input_ids, use_cache=False)
    return input_ids, position, recorder.activations[LAYER].detach()
activation_by_id = {p.prompt_id: activation_for_prompt(p.prompt_id) for pair in failures for p in (pair["short"], pair["long"])}
all_selected_ids = sorted(selected_ids)
vectors_by_id = {i: jlens_vector(lens, unembedding_weight, layer=LAYER, token_id=i) for i in all_selected_ids}

In [ ]:
def selected_for(pair, kind, k):
    problem_id = pair["problem_id"]
    return selections.query("problem_id == @problem_id and kind == @kind").head(k).token_id.astype(int).tolist()

def run_patch(source_prompt, target_prompt, token_ids, alpha):
    source_input, source_position, source_activation = activation_by_id[source_prompt.prompt_id]
    _, target_position, target_activation = activation_by_id[target_prompt.prompt_id]
    selected_vectors = torch.stack([vectors_by_id[i] for i in token_ids]).T
    source_coordinates = lens_coordinates(source_activation, selected_vectors)
    target_at_position = lens_coordinates(target_activation[:, target_position, :], selected_vectors)
    target_coordinates = source_coordinates.clone()
    target_coordinates[:, source_position, :] = target_at_position
    with torch.inference_mode(), LensCoordinatePatcher(model.layers, {LAYER: selected_vectors}, {LAYER: target_coordinates}, alpha=alpha):
        logits = causal_lm(input_ids=source_input, use_cache=False).logits[0, -1]
    return next_token_score(logits, source_prompt)

summary_rows = []
for pair in failures:
    for direction, kind, source, target in (("restore_short_in_long", "lost", pair["long"], pair["short"]), ("inject_long_in_short", "gained", pair["short"], pair["long"])):
        baseline = pair["long_baseline"] if direction == "restore_short_in_long" else pair["short_baseline"]
        for k in K_VALUES:
            token_ids = selected_for(pair, kind, k)
            for alpha in ALPHAS:
                result = baseline if alpha == 0.0 or not token_ids else run_patch(source, target, token_ids, alpha)
                summary_rows.append({"problem_id": pair["problem_id"], "direction": direction, "k": k, "alpha": alpha, "baseline_prediction": baseline["prediction"], "baseline_score": baseline["score"], "intervened_prediction": result["prediction"], "intervened_score": result["score"], "target_rank": result["target_rank"]})
summary = pd.DataFrame(summary_rows)
summary["prediction_changed"] = summary.baseline_prediction != summary.intervened_prediction
summary["helped"] = (summary.baseline_score == 0) & (summary.intervened_score == 1)
summary["hurt"] = (summary.baseline_score == 1) & (summary.intervened_score == 0)
aggregate = summary.groupby(["direction", "k", "alpha"], as_index=False).agg(baseline_accuracy=("baseline_score", "mean"), intervention_accuracy=("intervened_score", "mean"), number_helped=("helped", "sum"), number_hurt=("hurt", "sum"), predictions_changed=("prediction_changed", "sum"))
display(aggregate)

## Readable examples and causal cases

In [ ]:
def concept_text(pair, kind, k=10):
    problem_id = pair["problem_id"]
    return selections.query("problem_id == @problem_id and kind == @kind").head(k)[["token", "short_rank", "short_score", "long_rank", "long_score"]].to_dict("records")
example_rows = [{"problem_id": p["problem_id"], "selected_lost_in_long": concept_text(p, "lost"), "selected_gained_in_long": concept_text(p, "gained")} for p in failures[:5]]
display(pd.DataFrame(example_rows))

fixed = summary.query("direction == 'restore_short_in_long' and intervened_score == 1 and baseline_score == 0").merge(selections.query("kind == 'lost'")[['problem_id', 'selection_rank', 'token']], on='problem_id', how='left')
fixed = fixed[fixed.selection_rank <= fixed.k]
broken = summary.query("direction == 'inject_long_in_short' and intervened_score == 0 and baseline_score == 1").merge(selections.query("kind == 'gained'")[['problem_id', 'selection_rank', 'token']], on='problem_id', how='left')
broken = broken[broken.selection_rank <= broken.k]
print("Cases where restoring short concepts fixed the long prompt")
display(fixed[['problem_id', 'k', 'alpha', 'token', 'intervened_prediction']].drop_duplicates())
print("Cases where injecting long concepts broke the short prompt")
display(broken[['problem_id', 'k', 'alpha', 'token', 'intervened_prediction']].drop_duplicates())

## Interpretation

The aggregate table and case tables above answer the causal questions. Evidence for both directions requires that restoring lost concepts sometimes helps long prompts and injecting gained concepts sometimes hurts short prompts; otherwise the corresponding direction is not supported by this intervention.